# OpenAI SDK Structured Output

source: [structured outputs guide](https://developers.openai.com/api/docs/guides/structured-outputs)

In this example we'll use the sync client, you can also use the async client.

Under the hood the SDK generates a JSON Schema from the Pydantic model and sends it to the API. The SDK guarantees the output structure only if the provider supports constrained decoding (strict: true). In any case the SDK will try to parse the output and raise an error if it's not parseable.

Constrained decoding: the JSON Schema is compiled into a state machine, and at each step a deterministic parser masks the logits of the tokens that would be invalid, so they can't be chosen.

Gotchas (hard-learned):
- Pydantic guarantees the structure, not the meaning. I received:
    - a refusal hidden in the payment_description field
    - a parsed payment_description with zero detected invoices but 100% confidence (the meaning for the LLMs was that it was sure that there were no invoice in that payment description but i expected 0% because human should be alerted) 
- You must instruct the LLM on what to do when the input doesn't match the expectations otherwise it is probable that it will hallucinate
- You must check the output ids match the input ids if you provide a batched input and you expect the LLM to return an output item for each input item
- A well-described output structure is enough to replace a lot of the content of a prompt (use Field(description=...) to improve results)
- You can use literals to create categories


## Setup data and system prompt

In [9]:
from dotenv import load_dotenv

# to add .env content to environment variables
load_dotenv()

payment_descriptions = [
    { "id": 1, "payment_description": "PAG. 2 ACC. FATT. N. 999 DEL 31 01 2026" },
    { "id": 2, "payment_description": "SALDO FATTURA N. 6549 DEL 30 04 2026" },
    { "id": 3, "payment_description": "SALDO FATT. NR 455 DEL 31 1 2026 - 765 DEL 28 22026 E 830 DEL 31 3 2026" },
    { "id": 4, "payment_description": "PAGAMENTO FATTURA N 875 DEL 29 06 2026 DDT 3432 DEL 10 05 2026" },
    { "id": 5, "payment_description": "Contoso SRL - RIF. 64 28 02 2026, 20 31 01 2026, 2398 28 10 2025, 3983 30 11 2025" },
    { "id": 6, "payment_description": "SALDO VS. FATTURE NR. 354 + 34 +12" },
    { "id": 7, "payment_description": "etichette per tracciabilita grano" },
    { "id": 8, "payment_description": "138 31 3 26" },
    { "id": 9, "payment_description": "55 28 2" },
    { "id": 10, "payment_description": "saldo fatture 2025" },
    { "id": 11, "payment_description": "saldo fatture da nr 100 a nr 300" },
    { "id": 12, "payment_description": "" },
    { "id": 13, "payment_description": "non pago nulla" }
]

## Request without Pydantic

In [36]:
import json
from openai import OpenAI
from openai.types import Reasoning

# payment_description should not be returned by the model because it's a waste of token but to simplify the reading i'll include it here 
SYSTEM_PROMPT = """
    you are a classifier of Italian bank credit transfer descriptions (causali bonifico).
    you can assume language is italian.
    you must return a JSON array containing an object per input object with the following fields:
     - id: the number of the input item (must match the input and it is mandatory) 
     - payment_description: the original payment_description
     - invoices: list of objects containing, if available: 
        - "invoice_number": the invoice number, it's a number
        - "date": YYYY-MM-DD  
     - confidence: confidence level between 0 and 1
     - notes: help a human understand how you extracted the data if they were not explicit, this is optional and you should only add it if data is ambiguous
    if you cannot find any invoice in a payment description write "could not find any invoice" in the notes.
    """

client = OpenAI() 

response = client.responses.create(
    model="gpt-5-nano",
    instructions=SYSTEM_PROMPT,
    input=json.dumps(payment_descriptions, indent=2),
    reasoning=Reasoning(effort="minimal"),
    store=False  
)

# response.output_text is probably a json that we should parse 

print(f"usage - input tokens: {response.usage.input_tokens} - output tokens: {response.usage.output_tokens} - total: {response.usage.total_tokens}")
print(f"\noutput text: \n{response.output_text}")

usage - input tokens: 639 - output tokens: 1244 - total: 1883

output text: 
[
  {
    "id": 1,
    "payment_description": "PAG. 2 ACC. FATT. N. 999 DEL 31 01 2026",
    "invoices": [
      {
        "invoice_number": 999,
        "date": "2026-01-31"
      }
      ],
    "confidence": 0.75,
    "notes": "Formato chiaro: numero fattura 999 e data 31-01-2026."
  },
  {
    "id": 2,
    "payment_description": "SALDO FATTURA N. 6549 DEL 30 04 2026",
    "invoices": [
      {
        "invoice_number": 6549,
        "date": "2026-04-30"
      }
      ],
    "confidence": 0.75,
    "notes": "Indica saldo fattura con numero 6549 e data 30-04-2026."
  },
  {
    "id": 3,
    "payment_description": "SALDO FATT. NR 455 DEL 31 1 2026 - 765 DEL 28 22026 E 830 DEL 31 3 2026",
    "invoices": [
      {
        "invoice_number": 455,
        "date": "2026-01-31"
      },
      {
        "invoice_number": 765,
        "date": "2026-02-28"
      },
      {
        "invoice_number": 830,
        "date":

## Request with pedantic and guard rails

In [34]:
from pydantic import BaseModel
class RefusalError(Exception): ...
class IncompleteResponseError(Exception): ...


class Invoice(BaseModel):
    invoice_number: int | None
    date: str | None
class Classification(BaseModel):
    id: int
    payment_description: str
    invoices: list[Invoice]
    confidence: float
    notes: str | None
class Batch(BaseModel):
    results: list[Classification]

# payment_description should not be returned by the model because it's a waste of token but to simplify the reading i'll include it here 
SYSTEM_PROMPT = """
    you are a classifier of Italian bank credit transfer descriptions (causali bonifico).
    you can assume language is italian.
    if you cannot find any invoice in a payment description write "could not find any invoice" in the notes.
    """

client = OpenAI() 

# Here we use .PARSE
response = client.responses.parse(
    model="gpt-5-nano", # in this case gpt-5.6-terra gave otpimal results, gpt-5-nano results were not accurate
    instructions=SYSTEM_PROMPT,
    input=json.dumps(payment_descriptions, indent=2),
    text_format=Batch,
    reasoning=Reasoning(effort="minimal"),
    store=False  
)

# response.output_parsed is already parsed as an instance of Batch

# handle incomplete response due to max output tokens
if response.status == "incomplete":
    raise IncompleteResponseError(str(response.incomplete_details))

# refusal arrives as a content block of type "refusal"
# note: in my tests this never triggered — the model put the refusal text
# into payment_description instead, and the output still validated
for output in response.output:
    if output.type == "message":
        for item in output.content:
            if item.type == "refusal":
                raise RefusalError(item.refusal)

# handle refusal response due to security reasons - note that in my tests this has never been triggered because the LLM placed the refusal message in the payment_description field of the output.
for output in response.output:
    if output.type != "message":
        continue
    for item in output.content:
        if item.type == "refusal":
            print(item.refusal)
            raise Exception("Refusal")
        if not item.parsed:
            raise Exception("Could not parse response")
# guardrails on number of ids in input vs ids in output
ids_in_input = {d["id"] for d in payment_descriptions}
ids_in_output = {r.id for r in response.output_parsed.results}
missing = ids_in_input - ids_in_output
unexpected = ids_in_output - ids_in_input
duplicated = len(response.output_parsed.results)-len(ids_in_output)
print(f"inputs: {len(ids_in_input)} - outputs: {len(ids_in_output)}")
if (len(missing) > 0):
    print(f"missing: {sorted(missing)}")
if (len(unexpected) > 0):
    print(f"unexpected: {sorted(unexpected)}")
if (duplicated > 0):
    print(f"there are {duplicated} duplicates")

print(f"usage - input tokens: {response.usage.input_tokens} - output tokens: {response.usage.output_tokens} - total: {response.usage.total_tokens}")
print(f"\noutput parsed: \n{response.output_parsed.model_dump_json(indent=2)}")

inputs: 13 - outputs: 13
usage - input tokens: 699 - output tokens: 5924 - total: 6623

output parsed: 
{
  "results": [
    {
      "id": 1,
      "payment_description": "PAG. 2 ACC. FATT. N. 999 DEL 31 01 2026",
      "invoices": [
        {
          "invoice_number": 999,
          "date": "31 01 2026"
        }
      ],
      "confidence": 0.92,
      "notes": null
    },
    {
      "id": 2,
      "payment_description": "SALDO FATTURA N. 6549 DEL 30 04 2026",
      "invoices": [
        {
          "invoice_number": 6549,
          "date": "30 04 2026"
        }
      ],
      "confidence": 0.92,
      "notes": null
    },
    {
      "id": 3,
      "payment_description": "SALDO FATT. NR 455 DEL 31 1 2026 - 765 DEL 28 22026 E 830 DEL 31 3 2026",
      "invoices": [
        {
          "invoice_number": 455,
          "date": "31 01 2026"
        },
        {
          "invoice_number": 765,
          "date": "28 02 2026"
        },
        {
          "invoice_number": 830,
     